# **ΠΛΗΡΟΦΟΡΙΕΣ ΟΜΑΔΑΣ:**
ΒΕΝΕΤΟΠΟΥΛΟΣ ΑΡΙΣΤΟΤΕΛΗΣ 1115202000266

ΛΥΚΟΥΡΓΙΩΤΗΣ ΔΗΜΗΤΡΙΟΣ 1115202000257

ΜΠΑΝΤΑΚ ΕΜΙΛ 1115201800122


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install transformers torch langdetect pandas




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using ca

In [ ]:
!pip install transformers torch langdetect pandas

import os
import pandas as pd
import re
from transformers import pipeline, AutoTokenizer
import torch
from langdetect import detect

# Define required columns (excluding 'id')
required_columns = ['comments']

# Define lists of negative and positive words
negative_words = set(['terrible', 'awful', 'gross', 'dirty', 'disappointed', 'bad', 'horrible', 'worst', 'Cramped','Dirty','Uncomfortable','Dated','Noisy','Run-down','Stuffy','Unkempt','Bad view','Inconvenient','Dingy','Shabby','Poorly maintained','Bland','Smelly','Dark','Cluttered','Basic','Dreary','Uninviting','Drawback','Firm beds','Inconsistent water heat'])
positive_words = set(['great', 'amazing', 'wonderful', 'clean', 'happy', 'good', 'best', 'Wonderful', 'Fully equipped','Beautiful decor','Comfortable','Very clean','Spacious','Neat','Organized','Quiet','Convenient','Helpful','Excellent host','Accessible','Safe','Impeccable','Gracious','Perfect','Well located','Bright','Modern','Inviting','Professional','Friendly','Thoughtful','Responsive','Polite','Great amenities','Stylish','Efficient','Welcoming'])

# Function to preprocess text
def preprocess_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # Remove links
    text = re.sub(r'\@\w+|\#', '', text)  # Remove hashtags and mentions
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    return text

# Function to check for negative words
def contains_negative_words(text):
    return any(word in text.split() for word in negative_words)

# Function to check for positive words
def contains_positive_words(text):
    return any(word in text.split() for word in positive_words)

# Function to detect if a comment is in English
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

# Load data from the folder and preprocess it
def process_folder(folder_path, output_filename):
    # Define the model and tokenizer
    model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
    sentiment_pipeline = pipeline("sentiment-analysis", model=model_name, tokenizer=model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dfs = []
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            for filename in os.listdir(subfolder_path):
                if filename.endswith("reviews.csv"):  # Specify the appropriate data file
                    file_path = os.path.join(subfolder_path, filename)
                    df = pd.read_csv(file_path, dtype=str)  # Read data as strings
                    # Check if 'comments' column exists in the DataFrame
                    if 'comments' not in df.columns:
                        print(f"The file {filename} does not contain the 'comments' column. Skipping.")
                        continue
                    # Filter to keep only the required columns (excluding 'id')
                    df_filtered = df[[col for col in required_columns if col in df.columns]].copy()  # Explicit copy
                    dfs.append(df_filtered)

    if not dfs:
        print(f"No suitable CSV files found in the folder: {folder_path}")
        return

    # Merge DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True, sort=False).copy()  # Explicit copy to avoid SettingWithCopyWarning

    # Drop rows with NaN values in 'comments' column
    combined_df.dropna(subset=['comments'], inplace=True)

    # Sample 10,000 random comments
    sampled_df = combined_df.sample(n=15000, random_state=42)

    # Filter for English comments
    sampled_df['is_english'] = sampled_df['comments'].apply(is_english)
    english_comments_df = sampled_df[sampled_df['is_english']].drop(columns=['is_english'])

    # Preprocess the 'comments' column
    english_comments_df['comments'] = english_comments_df['comments'].apply(preprocess_text)

    # Define sentiment mapping
    sentiment_mapping = {
        '1 star': 'negative',
        '2 stars': 'negative',
        '3 stars': 'neutral',
        '4 stars': 'positive',
        '5 stars': 'positive'
    }

    # Initialize sentiment counts
    sentiment_counts = {'negative': 0, 'neutral': 0, 'positive': 0}

    # Define sentiment limits
    sentiment_limits = {'negative': 200, 'neutral': 200, 'positive': 600}

    # Initialize results list
    results = []
    sentiment_id = 1  # Initialize sentiment ID counter

    # Process in chunks for better performance
    chunk_size = 100  # Adjust the chunk size as needed
    max_length = 512  # Maximum length for tokenized input

    for i in range(0, len(english_comments_df), chunk_size):
        chunk = english_comments_df[i:i + chunk_size]
        truncated_comments = []
        for comment in chunk['comments']:
            tokenized_comment = tokenizer(comment, truncation=True, max_length=max_length, padding=False)
            truncated_comments.append(tokenizer.decode(tokenized_comment['input_ids'], skip_special_tokens=True))

        chunk['comments'] = truncated_comments

        # Tokenize inputs and ensure they don't exceed the maximum sequence length
        inputs = tokenizer(chunk['comments'].tolist(), truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}  # Move inputs to the appropriate device

        # Perform sentiment analysis on the inputs
        for j, comment in enumerate(chunk['comments']):
            if contains_negative_words(comment):
                sentiment_label = 'negative'
            else:
                sentiment = sentiment_pipeline(comment)[0]  # Perform sentiment analysis on the comment
                sentiment_label = sentiment_mapping[sentiment['label']]  # Map to positive, negative, neutral

            if sentiment_counts[sentiment_label] < sentiment_limits[sentiment_label]:
                results.append({
                    'id': sentiment_id,
                    'review': comment,
                    'sentiment': sentiment_label
                })
                sentiment_counts[sentiment_label] += 1
                sentiment_id += 1  # Increment sentiment ID

            # Break if we have enough reviews for each sentiment
            if all(sentiment_counts[sentiment] >= sentiment_limits[sentiment] for sentiment in sentiment_counts):
                break
        else:
            continue
        break

    # Create a new DataFrame with the results
    results_df = pd.DataFrame(results)

    # Save the results to a CSV file
    results_df.to_csv(output_filename, index=False, mode='a', header=not os.path.exists(output_filename))

    print("Sentiment analysis completed and the results have been saved.")
    print("Final sentiment counts:")
    print(sentiment_counts)

# Process folders for 2019 and 2023
process_folder('/content/drive/My Drive/data/2019', '/content/drive/My Drive/data/sentiment_results_2019.csv')
#process_folder('/content/drive/My Drive/data/2023', '/content/drive/My Drive/data/sentiment_results_2023.csv')


<ipython-input-6-9f37ec673ac3>:115: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chunk['comments'] = truncated_comments


Sentiment analysis completed and the results have been saved.
Final sentiment counts:
{'negative': 200, 'neutral': 200, 'positive': 600}


# **The second part of the project begins from here**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/dataset_tede/data/sentiment_results_2019.csv')
print(df.shape)

(1000, 3)


In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)

# Save the training and testing sets to TSV files
train.to_csv('/content/drive/MyDrive/dataset_tede/data/train.tsv', sep='\t', index=False)
test.to_csv('/content/drive/MyDrive/dataset_tede/data/test.tsv', sep='\t', index=False)

In [ ]:
df_train = pd.read_csv('/content/drive/MyDrive/dataset_tede/data/train.tsv', sep='\t')

df_test = pd.read_csv('/content/drive/MyDrive/dataset_tede/data/test.tsv', sep='\t')

In [ ]:
from collections import Counter
import numpy as np
from numpy.linalg import norm

words_count = Counter(" ".join(df_train["review"]).split()).most_common(500)

# Extracting the first element from each tuple
Vocabulary = [tup[0] for tup in words_count]
# Vocabulary is a list with the 500 most common words which they appear on the column "review" from the dataset df_train
print(Vocabulary)

['the', 'and', 'to', 'a', 'was', 'is', 'in', 'we', 'very', 'for', 'of', 'apartment', 'it', 'with', 'i', 'you', 'great', 'us', 'place', 'this', 'stay', 'but', 'location', 'not', 'athens', 'host', 'from', 'there', 'at', 'our', 'nice', 'were', 'on', 'had', 'that', 'as', 'clean', 'all', 'so', 'are', 'good', 'would', 'an', 'everything', 'also', 'have', 'really', 'if', 'metro', 'its', 'close', 'which', 'be', 'recommend', 'walk', 'he', 'perfect', 'well', 'my', 'flat', 'just', 'time', 'comfortable', 'by', 'area', 'helpful', 'acropolis', 'restaurants', 'walking', 'only', 'out', 'one', 'some', 'about', 'like', 'has', 'room', 'night', 'when', 'need', 'highly', 'amazing', 'can', 'she', 'easy', 'even', 'station', 'get', 'his', 'no', 'or', 'city', 'her', 'located', 'around', 'here', 'they', 'again', 'up', 'more', 'kitchen', 'than', 'view', 'me', 'arrival', 'before', 'could', 'didnt', 'distance', 'communication', 'much', 'other', 'will', 'house', 'definitely', 'places', 'lovely', 'airport', 'your', '

In [ ]:
import gensim
from gensim.models import FastText

# Initialize and train a model:

corpus = [[word] for word in Vocabulary]

model = FastText(vector_size=100, window=3, min_count=1)  # instantiate

model.build_vocab(corpus_iterable=corpus)

model.train(corpus_iterable=corpus, total_examples=len(corpus), epochs=10)  # train

vocab = list(model.wv.key_to_index.keys())
# print(vocab)

# print('the' in model.wv.key_to_index)

# model.wv["the"]

In [ ]:
# We create a new column on each dataset "filtered_review" that has words from the Vocabulary that they have appeared
# on the column "review" and this procedure it is done for every row from each dataset
def filter_words(text, vocabulary):
    words = text.split()
    return " ".join([word for word in words if word in vocabulary])

df_train["filtered_review"] = df_train["review"].apply(lambda x: filter_words(x, Vocabulary))

df_test["filtered_review"] = df_test["review"].apply(lambda x: filter_words(x, Vocabulary))

In [ ]:
# Tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

tfidf_vectorizer = TfidfVectorizer()

train_tfidf = tfidf_vectorizer.fit_transform(df_train['filtered_review'])
# print(train_tfidf)
# print("\n")

test_tfidf = tfidf_vectorizer.transform(df_test['filtered_review'])
# print(test_tfidf)

# Save train_tfidf and test_tfidf to .pkl files
with open('train_tfidf.pkl', 'wb') as f:
    pickle.dump(train_tfidf, f, protocol=pickle.HIGHEST_PROTOCOL)

with open('test_tfidf.pkl', 'wb') as f:
    pickle.dump(test_tfidf, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Word embeddings with fastText
def Word_embeddings_fastText(text, model):
    words = text.split()
    word_vectors = [model.wv[word] for word in words if word in model.wv]
    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(word_vectors, axis=0)

print(model)
print(model.wv)
train_ft = np.array([Word_embeddings_fastText(text, model) for text in df_train['filtered_review']])
# print(train_ft)

test_ft = np.array([Word_embeddings_fastText(text, model) for text in df_test['filtered_review']])

# Save train_ft and test_ft to .pkl files
with open('train_ft.pkl', 'wb') as f:
    pickle.dump(train_ft, f, protocol=pickle.HIGHEST_PROTOCOL)

with open('test_ft.pkl', 'wb') as f:
    pickle.dump(test_ft, f, protocol=pickle.HIGHEST_PROTOCOL)

FastText<vocab=500, vector_size=100, alpha=0.025>
FastTextKeyedVectors<vector_size=100, 500 keys>


In [ ]:
# For Tf-idf
# Load .pkl files
with open('train_tfidf.pkl', 'rb') as f:
    train_tfidf = pickle.load(f)

with open('test_tfidf.pkl', 'rb') as f:
    test_tfidf = pickle.load(f)

# For word embeddings
# Load .pkl files
with open('train_ft.pkl', 'rb') as f:
    train_ft = pickle.load(f)

with open('test_ft.pkl', 'rb') as f:
    test_ft = pickle.load(f)

# print(train_ft)

In [ ]:
# for svm classification
from sklearn.svm import SVC

# for Random forest classification
from sklearn.ensemble import RandomForestClassifier

# for knn classification
from sklearn.neighbors import KNeighborsClassifier

# 10-fold Cross Validation
from sklearn.model_selection import cross_val_score, KFold

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# for time measure
from timeit import default_timer as timer

def my_classifier(clf, sub_method, train, test):
  start_timer = timer()
  codelabel_encoder = LabelEncoder()

  label_train = codelabel_encoder.fit_transform(df_train["sentiment"])
  label_test =  codelabel_encoder.transform(df_test["sentiment"])

  if clf == 'svm':
      classifier = SVC(kernel='linear', C=1, class_weight='balanced')
  elif clf == 'RandomForest':
      classifier = RandomForestClassifier(class_weight='balanced', n_estimators=50, random_state=42)
  elif clf == 'knn':
      classifier = KNeighborsClassifier(n_neighbors=5)

  # here the separation is done
  kf = KFold(n_splits=10, shuffle=True, random_state=42)

  print("----------------------------------------------------")
  print("Outputs for the classifier: " + clf + ", with the method: " + sub_method + "\n")

  # model scores
  # Evaluate the model using cross_val_score
  scores = cross_val_score(classifier, train, label_train, cv=kf)
  print(f"Cross-validation scores: {scores}")
  print(f"Mean accuracy: {scores.mean():.2f}")
  print(f"Standard deviation: {scores.std():.2f}")
  print("\n")

  true_labels = []
  predicted_labels = []

  for train_index, test_index in kf.split(train):
    X_train, X_test = train[train_index], train[test_index]
    y_train, y_test = label_train[train_index], label_train[test_index]

    classifier.fit(X_train, y_train)
    predictions = classifier.predict(X_test)

    true_labels.extend(y_test)
    predicted_labels.extend(predictions)


  true_labels = codelabel_encoder.inverse_transform(true_labels)
  predicted_labels = codelabel_encoder.inverse_transform(predicted_labels)
  #classifier scores
  report = classification_report(true_labels, predicted_labels)
  print(report)
  # print(set(true_labels) - set(predicted_labels))
  print("\n")

  # Final evaluation for the test_part
  test_score = classifier.score(test, label_test)
  print("Final results of the " + clf + " for the test_part: ", test_score)
  print("\n")

  end_timer = timer()
  print("With the total time of " + clf + " with " + sub_method + ":", end_timer - start_timer)
  print("----------------------------------------------------")

# For svm with Tf-idf
my_classifier("svm", "Tf-idf", train_tfidf, test_tfidf)

# For svm with Word embeddings
my_classifier("svm", "Word embeddings", train_ft, test_ft)

# For RandomForest with Tf-idf
my_classifier("RandomForest", "Tf-idf", train_tfidf, test_tfidf)

# For RandomForest with Word embeddings
my_classifier("RandomForest", "Word embeddings", train_ft, test_ft)

# For knn with Tf-idf
my_classifier("knn", "Tf-idf", train_tfidf, test_tfidf)

# For knn with Word embeddings
my_classifier("knn", "Word embeddings", train_ft, test_ft)

----------------------------------------------------
Outputs for the classifier: svm, with the method: Tf-idf

Cross-validation scores: [0.7625 0.725  0.825  0.725  0.7125 0.825  0.8    0.775  0.8    0.7375]
Mean accuracy: 0.77
Standard deviation: 0.04


              precision    recall  f1-score   support

    negative       0.64      0.63      0.63       159
     neutral       0.56      0.62      0.59       154
    positive       0.89      0.86      0.87       487

    accuracy                           0.77       800
   macro avg       0.69      0.70      0.70       800
weighted avg       0.78      0.77      0.77       800



Final results of the svm for the test_part:  0.73


With the total time of svm with Tf-idf: 4.451048986000018
----------------------------------------------------
----------------------------------------------------
Outputs for the classifier: svm, with the method: Word embeddings

Cross-validation scores: [0.6375 0.15   0.6625 0.225  0.1125 0.2125 0.6625 0.6 

## **OUTPUT REPORT:**

The evaluation will be done in terms of the:

precision = How many of the reviews we said were in one category actually are in that category

recall = How many of the reviews that are actually in a category did we correctly identify

f1-score = Is the overall score between the precision and the recall parts

macro avg = it is generally how it went for all the categories combine (i.e. negative, neutral, positive) in each subcategory (i.e. for the part of the precision, recall, f1-score)

accuracy = it is for the three categories how it went overall (i.e. is equivalent with the overall score)

Before we compare the classifiers its important to say that for the classifiers svm and RandomForest we pass them as a parameter the class_weight='balanced'

*   Precision:

      1) The best is RandomForest with Tf-idf has very good scores in the categories of negative and neutral compared to the others classifiers but the score for positives is not the best (the best for this category is from the svm with Tf-idf) but compare to the others classifiers expect of the svm with Tf-idf has the best score for the category positive

      2) In the second place we have two the RandomForest with Word embeddings and the svm with Tf-idf. The score for the negative category the RandomForest with Word embeddings has much better score, the score for the neutrals the two they don't have a huge difference but the score for the positives the svm with Tf-idf is better from all the others classifiers
      
      3) Third is the knn (with n = 5) with Tf-idf

      4) After that we have the knn (with n = 5) with Word embeddings

      5) And the worst score for the precision is from the svm with Word embeddings
        
*   Recall:

  1) The best is svm with Tf-idf it has the best by far scores for the categories of negative and neutral compared to the others classifiers but the score for positives is not the best, the best for this category are from the RandomForest with Tf-idf and Word embeddings but the difference between the three of them for this category its not huge    

  2) In the second place we have the RandomForest with Tf-idf

  3) After that we have the knn (with n = 5) with Tf-idf

  4) In this place we have the knn (with n = 5) with Word embeddings

  5) RandomForest with Word embeddings, but for the category positive this classifier has the best score compare with all the others classifiers on this category

  6) And the worst score for the recall is from the svm with Word embeddings

*   f1-score:
  
  1) The best is the svm with Tf-idf

  2) In the second place we have the RandomForest with Tf-idf

  3) After that we have the the knn (with n = 5) with Tf-idf

  4) RandomForest with Word embeddings

  5) In this place we have the knn (with n = 5) with Word embeddings

  6) And the worst score for the f1-score is from the svm with Word embeddings

*   macro avg:

  1) First in this part is the classifier svm with Tf-idf

  2) After that we have the RandomForest with Tf-idf

  3) RandomForest with Word embeddings

  4) In this place we have the knn (with n = 5) with Tf-idf

  5) knn (with n = 5) with Word embeddings

  6) And the worst classifier is the svm with Word embeddings

*   Accuracy ( = i.e. overall score):

  1) Overall in the first place we have the svm with Tf-idf classifier

  2) After that and not by a huge difference is the RandomForest with Tf-idf

  3) In this place we have the RandomForest with Word embeddings

  4) knn (with n = 5) with Tf-idf

  5) knn (with n = 5) with Word embeddings

  6) And overall the worst classifier by far is the svm with Word embeddings


# **The third part is bellow**

In [6]:
import os
import pandas as pd
import re
import random
from langdetect import detect
from gensim.downloader import load as gensim_load
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr
import numpy as np

# Define required columns (excluding 'id')
required_columns = ['comments']

# Function to preprocess text
def preprocess_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()  # Convert to lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)  # Remove links
    text = re.sub(r'\@\w+|\#', '', text)  # Remove hashtags and mentions
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = re.sub(r'\d+', '', text)  # Remove numbers
    return text

# Function to detect if a comment is in English
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

# Load pre-trained Word2Vec embeddings
print("Loading Word2Vec model...")
model = gensim_load('word2vec-google-news-300')
print("Model loaded.")

# Function to calculate cosine similarity
def cosine_similarity(vec1, vec2):
    return 1 - cosine(vec1, vec2)

# Function to get the neighborhood of a word
def get_neighborhood(word, N):
    return model.most_similar(word, topn=N)

# Function to calculate neighborhood similarity using three methods
def neighborhood_similarity(word1, word2, N):
    neighbors1 = get_neighborhood(word1, N)
    neighbors2 = get_neighborhood(word2, N)

    neighbors1_vectors = [model[neighbor] for neighbor, _ in neighbors1]
    neighbors2_vectors = [model[neighbor] for neighbor, _ in neighbors2]

    max_similarity = max(cosine_similarity(model[word1], vec2) for vec2 in neighbors2_vectors)

    similarities1 = [cosine_similarity(model[word1], vec1) for vec1 in neighbors1_vectors]
    similarities2 = [cosine_similarity(model[word2], vec2) for vec2 in neighbors2_vectors]
    correlation, _ = pearsonr(similarities1, similarities2)

    sum_squared_similarities = np.sum((np.array(similarities1) - np.array(similarities2)) ** 2)

    return max_similarity, correlation, sum_squared_similarities

# Function to select random words from comments
def get_random_words(comments, num_words=2):
    words = []
    for comment in comments:
        words.extend(comment.split())
    return random.sample(words, num_words)

# Load data from the folder and preprocess it
def process_folder(folder_path, output_filename, num_words=2, N=10):
    dfs = []
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            for filename in os.listdir(subfolder_path):
                if filename.endswith("reviews.csv"):  # Specify the appropriate data file
                    file_path = os.path.join(subfolder_path, filename)
                    df = pd.read_csv(file_path, dtype=str)  # Read data as strings
                    if 'comments' not in df.columns:
                        print(f"The file {filename} does not contain the 'comments' column. Skipping.")
                        continue
                    df_filtered = df[[col for col in required_columns if col in df.columns]].copy()  # Explicit copy
                    dfs.append(df_filtered)

    if not dfs:
        print(f"No suitable CSV files found in the folder: {folder_path}")
        return

    combined_df = pd.concat(dfs, ignore_index=True, sort=False).copy()  # Explicit copy to avoid SettingWithCopyWarning

    combined_df.dropna(subset=['comments'], inplace=True)

    sampled_df = combined_df.sample(n=15000, random_state=42)

    sampled_df['is_english'] = sampled_df['comments'].apply(is_english)
    english_comments_df = sampled_df[sampled_df['is_english']].drop(columns=['is_english'])

    english_comments_df['comments'] = english_comments_df['comments'].apply(preprocess_text)

    random_words = get_random_words(english_comments_df['comments'], num_words)
    word1, word2 = random_words[0], random_words[1]

    display_neighborhood(word1, N)
    display_neighborhood(word2, N)

    max_sim, corr, sum_sq_sim = neighborhood_similarity(word1, word2, N)
    print(f"\nSelected words: {word1}, {word2}")
    print(f"Max similarity of neighborhoods: {max_sim:.4f}")
    print(f"Correlation of neighborhood similarities: {corr:.4f}")
    print(f"Sum of squared neighborhood similarities: {sum_sq_sim:.4f}")

# Display neighborhood
def display_neighborhood(word, N):
    neighborhood = get_neighborhood(word, N)
    print(f"Neighborhood of {word}:")
    for neighbor, similarity in neighborhood:
        print(f"  {neighbor}: {similarity:.4f}")

# Process folders for 2019
process_folder('/content/drive/My Drive/data/2019', '/content/drive/My Drive/data/similarity_results_2019.csv')


Loading Word2Vec model...
Model loaded.
Neighborhood of area:
  areas: 0.6797
  outlying_areas: 0.5878
  region: 0.5685
  vicinity: 0.5611
  neighborhoods: 0.5483
  neighborhood: 0.5397
  Area: 0.5268
  metropolitan: 0.5027
  mile_radius: 0.4948
  vicinities: 0.4938
Neighborhood of heart:
  hearts: 0.5909
  Heart: 0.5590
  heartbeat: 0.5310
  congestive_heart_failure: 0.5162
  cardiac: 0.5147
  myocardial_infraction: 0.5028
  coronary_artery_clots: 0.5027
  lion_Frese: 0.4980
  empowers_CSPs: 0.4971
  coronary: 0.4948

Selected words: area, heart
Max similarity of neighborhoods: 0.2057
Correlation of neighborhood similarities: 0.9600
Sum of squared neighborhood similarities: 0.0152
